In [11]:
import datetime
import os
import uuid

from dotenv import load_dotenv
import xarray as xr

from xsdba.adjustment import QuantileDeltaMapping

In [2]:
load_dotenv()

ERA5_URI = os.environ["POREALLAS_PARSED_ERA5_URI"]
GMFD_URI = os.environ["POREALLAS_PARSED_GMFD_URI"]
OUT_ZARR = os.environ["POREALLAS_ERA5_URI"]
HISTREF_START_YEAR = 1981
HISTREF_STOP_YEAR = 1997
# 30-yr to 2045
SIM_START_YEAR = 2015
SIM_STOP_YEAR = 2045
QDM_N_QUANTILES = 10
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()

In [12]:
def lon_adjust(_ds, roll=True):
    if roll:
        _ds["longitude"] = (_ds["longitude"] + 180) % 360 - 180
    _ds = _ds.sortby("longitude")
    _ds = _ds.rename({"longitude": "lon", "latitude": "lat"})
    _ds = _ds.chunk("auto")
    return _ds

In [13]:
gmfd = xr.open_dataset(
    "/home/emily_zuetell/projects/poreallas/data/parsed/gmfd_parsed.zarr",
    engine="zarr",
    chunks={},
).groupby('time.year').mean()

# Fill extreme values
gmfd = gmfd.sortby("latitude").chunk({"latitude": -1, "longitude": 30, "year": -1})
gmfd = (
    gmfd.where(gmfd["tas"] < 1000)
    .interpolate_na(dim="latitude", method="linear")
    .compute()
)
gmfd = lon_adjust(gmfd, roll=True)

In [4]:
cmip6_sim = xr.open_dataset("/home/emily_zuetell/projects/poreallas/data/t_CMIP6_ssp370_mon_201501-210012.nc",
                                chunks={},
                                ).mean(dim = 'member').groupby('time.year').mean()
cmip6_hist = xr.open_dataset("/home/emily_zuetell/projects/poreallas/data/t_CMIP6_historical_mon_185001-201412.nc",
                                 chunks={},
                                 ).mean(dim = 'member').groupby('time.year').mean()

/home/emily_zuetell/miniforge3/envs/esmf-env/lib/python3.11/site-packages/xarray/conventions.py:205: SerializationWarning: variable 't' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/emily_zuetell/miniforge3/envs/esmf-env/lib/python3.11/site-packages/xarray/conventions.py:205: SerializationWarning: variable 't' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [5]:
cmip6_hist

<xarray.Dataset> Size: 44MB
Dimensions:    (year: 165, lat: 180, bnds: 2, lon: 360)
Coordinates:
  * year       (year) int64 1kB 1850 1851 1852 1853 1854 ... 2011 2012 2013 2014
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon        (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
    height2m   float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    lat_bnds   (year, lat, bnds) float64 475kB dask.array<chunksize=(1, 180, 2), meta=np.ndarray>
    lon_bnds   (year, lon, bnds) float64 950kB dask.array<chunksize=(1, 360, 2), meta=np.ndarray>
    time_bnds  (year, bnds) datetime64[ns] 3kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    t          (year, lat, lon) float32 43MB dask.array<chunksize=(1, 20, 40), meta=np.ndarray>
    crs        (year) float64 1kB -2.147e+09 -2.147e+09 ... -2.147e+09
Attributes: (12/28)
    Conventions:                CF-1.9 ACDD-1.3
    title:                      IPCC-WGI AR6 Interactive Atlas Dataset
    summary:                    IPCC-WGI AR6 Interactive Atlas dataset: Month...
    keywords:                   CMIP5, CMIP6, CORDEX, IPCC, Interactive Atlas
    institution:                Instituto de Fisica de Cantabria (IFCA, CSIC-...
    contact:                    ipcc-ddc@ifca.unican.es
    ...                         ...
    geospatial_lon_min:         -180.0
    geospatial_lon_max:         180.0
    geospatial_lon_resolution:  1.0
    geospatial_lon_units:       degrees_east
    date_created:               2022-10-26T00:00:00+00:00
    tracking_id:                4a14519f-05eb-4a2b-8e69-d5f1f8e500de

In [15]:
ref = gmfd.sel(year=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
hist = cmip6_hist.sel(year=slice(str(HISTREF_START_YEAR), str(HISTREF_STOP_YEAR)))
sim = cmip6_sim.sel(year=slice(str(SIM_START_YEAR), str(SIM_STOP_YEAR)))

In [16]:
def year_to_time(ds):
    ds = ds.rename({"year": "time"})
    ds["time"] = xr.date_range(
        start=f"{int(ds.time[0])}-01-01", periods=ds.sizes["time"],
        freq="YS", calendar="standard", use_cftime=True,
    )
    return ds.chunk({"time": -1})

ref = year_to_time(ref)
hist = year_to_time(hist)
sim = year_to_time(sim)

In [17]:
# # "time" dim cannot be chunked for QDM.
ref = ref.chunk({"time": -1})
hist = hist.chunk({"time": -1})
sim = sim.chunk({"time": -1})

qdm = QuantileDeltaMapping.train(
    ref["tas"], hist["t"], nquantiles=QDM_N_QUANTILES, kind="+", group="time"
)

In [18]:
sim_adj = qdm.adjust(sim["t"])

sim_adj.name = "tas"
sim_adj = sim_adj.to_dataset()

# Add additional general metadata.
sim_adj.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted ERA5 climate fields",
}
sim_adj["tas"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "QDM bias-adjusted ERA5 tas field",
    "poreallas_adjustment_method": "QDM",
    "poreallas_histref_start_year": HISTREF_START_YEAR,
    "poreallas_histref_stop_year": HISTREF_STOP_YEAR,
    "poreallas_sim_start_year": SIM_START_YEAR,
    "poreallas_sim_stop_year": SIM_STOP_YEAR,
    "poreallas_qdm_nquantiles": QDM_N_QUANTILES,
    "poreallas_ref_uri": GMFD_URI,
    "poreallas_hist_uri": ERA5_URI,
    "poreallas_sim_uri": ERA5_URI,
}

sim_adj = sim_adj.chunk("auto").compute()

In [20]:
sim_adj.to_zarr("cmip6_annual_adj.zarr", consolidated=True)

/home/emily_zuetell/miniforge3/envs/esmf-env/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
